<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Dataset Credits
</h2>

Dataset: **Health Insurance Dataset** by **Parth Patel**  
Source: https://www.kaggle.com/datasets/patelparth3399/indian-insurance-premium-prediction-dataset 

License: MIT

Special thanks to the author for making the dataset publicly available.

### ==========================================================
### Project : Insurance Premium Prediction API & Serving System
### Author  : Omi Kalix
### GitHub  : https://github.com/OMI-KALIX
### ==========================================================

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Insurance Premium Category Prediction using Random Forest
</h2>
This project builds a machine learning model to classify insurance premium categories (Low, Medium, High) based on customer demographics, lifestyle factors, and engineered health-related features.

We begin by importing the required Python libraries for data preprocessing, feature engineering, model training, evaluation, and model persistence.

In [65]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
from pickle import dump, load


<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Load Dataset
</h2>

Load the Indian Insurance Dataset into a Pandas DataFrame and inspect the first few records to understand its structure.

In [66]:
df = pd.read_csv('Indian_Insurance_Data.csv')
df.head()

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
0,50,57,176,29.90,False,Agra,Factory Worker,Medium
1,56,90,168,19.22,False,Allahabad,Businessman,Medium
2,26,87,178,36.95,False,Srinagar,Sales Manager,Low
3,34,73,162,14.75,False,Meerut,Banker,Low
4,43,47,192,19.32,True,Varanasi,Marketing Manager,High


<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Create Working Copy
</h2>

Create a copy of the original dataset to preserve the raw data and perform transformations safely.

In [67]:
df1=df.copy()

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Exploratory Data Analysis
</h2>

Analyze the dataset structure, data types, missing values, and descriptive statistics to gain initial insights.

In [68]:
df1.head()
df1.info()
df1.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   age                         4000 non-null   int64  
 1   weight                      4000 non-null   int64  
 2   height                      4000 non-null   int64  
 3   income_lpa                  4000 non-null   float64
 4   smoker                      4000 non-null   bool   
 5   city                        4000 non-null   object 
 6   occupation                  4000 non-null   object 
 7   insurance_premium_category  4000 non-null   object 
dtypes: bool(1), float64(1), int64(3), object(3)
memory usage: 222.8+ KB


,age,weight,height,income_lpa
count,4000.000000,4000.00000,4000.00000,4000.000000
mean,40.953750,71.96900,172.57175,25.896793
std,13.439504,16.13698,13.27258,13.946143
min,18.000000,45.00000,150.00000,2.010000
25%,29.000000,58.00000,161.00000,13.727500
50%,41.000000,72.00000,173.00000,25.600000
75%,52.000000,86.00000,184.00000,38.035000
max,64.000000,99.00000,195.00000,50.000000


<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Explore Categorical Variables

</h2>

Inspect the unique values present in city, occupation, and insurance premium category columns.

In [69]:
print(df1['city'].unique())
print(df1['insurance_premium_category'].unique())
print(df1['occupation'].unique())


['Agra' 'Allahabad' 'Srinagar' 'Meerut' 'Varanasi' 'Hyderabad' 'Ahmedabad'
 'Chennai' 'Amritsar' 'Vadodara' 'Lucknow' 'Bhopal' 'Mumbai' 'Ludhiana'
 'Rajkot' 'Surat' 'Ghaziabad' 'Ranchi' 'Pune' 'Kanpur' 'Nagpur'
 'Faridabad' 'Bangalore' 'Jaipur' 'Kolkata' 'Nashik' 'Patna' 'Indore'
 'Delhi' 'Visakhapatnam']
['Medium' 'Low' 'High']
['Factory Worker' 'Businessman' 'Sales Manager' 'Banker'
 'Marketing Manager' 'Insurance Agent' 'HR Manager' 'Pharmacist' 'Teacher'
 'Software Engineer' 'Consultant' 'Driver' 'Shop Owner' 'Nurse'
 'Accountant' 'Government Employee' 'Architect' 'Engineer'
 'Real Estate Agent' 'Civil Servant' 'Plumber' 'Retail Manager' 'Chef'
 'Electrician' 'Carpenter' 'Doctor' 'Lab Technician' 'Data Analyst'
 'Lawyer' 'Content Writer']


<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
City Tier Classification
</h2>

Group cities into Tier 1, Tier 2, and Tier 3 categories based on their economic and urban development levels.

This feature helps capture regional differences that may influence insurance premiums.

In [70]:
tier1_cities = [
    "Delhi",
    "Mumbai",
    "Bangalore",
    "Chennai",
    "Hyderabad",
    "Kolkata",
    "Pune",
    "Ahmedabad"
]

tier2_cities = [
    "Jaipur",
    "Lucknow",
    "Indore",
    "Bhopal",
    "Nagpur",
    "Surat",
    "Vadodara",
    "Visakhapatnam",
    "Patna",
    "Kanpur",
    "Ludhiana",
    "Nashik",
    "Rajkot",
    "Faridabad",
    "Ghaziabad"
]

tier3_cities = [
    "Agra",
    "Allahabad",
    "Srinagar",
    "Meerut",
    "Varanasi",
    "Amritsar",
    "Ranchi"
]

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Encode Smoker Status

</h2>

Convert the smoker column into numerical format:

- Smoker = 1
- Non-Smoker = 0

This transformation allows the model to process smoking status effectively.

In [71]:
def smoker_status(smoker):
    return 1 if smoker == True else 0
df1["smoker"] = df1["smoker"].apply(smoker_status)

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">Create City Tier Feature

</h2>

Map each city to its corresponding tier category:

- Tier 1 → 1
- Tier 2 → 2
- Tier 3 → 3

This engineered feature provides a simplified representation of city characteristics.

In [72]:
def tier_cities(city):
    if city in tier1_cities:
        return 1
    elif city in tier2_cities:
        return 2
    else:
        return 3
    
df1["city_tier"] = df1["city"].apply(tier_cities)

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">Feature Engineering

</h2>

Create additional informative features that may improve predictive performance:

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">Engineered Features

</h2>

- BMI (Body Mass Index)
- Income per Age
- Weight-to-Height Ratio
- Income × BMI
- Smoker × Income
- Health Risk Index

These features help capture health, lifestyle, and financial relationships relevant to insurance premium determination.

In [73]:
# BMI
df1["height_m"] = df1["height"] / 100

df1["bmi"] = df1["weight"] / (df1["height_m"]**2)

# Income per Age
df1["income_per_age"] = df1["income_lpa"] / df1["age"]

# Weight Height Ratio
df1["weight_height_ratio"] = df1["weight"] / df1["height"]

# Income × BMI
df1["income_bmi"] = df1["income_lpa"] * df1["bmi"]

# Smoker × Income
df1["smoker_income"] = df1["smoker"] * df1["income_lpa"]

# Health Risk Index
df1["health_risk_index"] = (
    df1["bmi"] * 0.4 +
    df1["age"] * 0.3 +
    df1["smoker"] * 20
)


<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Preview Engineered Dataset

</h2>

Inspect the dataset after feature engineering to verify that all new features have been created correctly.

In [74]:
df1.head()

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,city_tier,height_m,bmi,income_per_age,weight_height_ratio,income_bmi,smoker_income,health_risk_index
0,50,57,176,29.90,0,Agra,Factory Worker,Medium,3,1.76,18.401343,0.598000,0.323864,550.200155,0.00,22.360537
1,56,90,168,19.22,0,Allahabad,Businessman,Medium,3,1.68,31.887755,0.343214,0.535714,612.882653,0.00,29.555102
2,26,87,178,36.95,0,Srinagar,Sales Manager,Low,3,1.78,27.458654,1.421154,0.488764,1014.597273,0.00,18.783462
3,34,73,162,14.75,0,Meerut,Banker,Low,3,1.62,27.815882,0.433824,0.450617,410.284255,0.00,21.326353
4,43,47,192,19.32,1,Varanasi,Marketing Manager,High,3,1.92,12.749566,0.449302,0.244792,246.321615,19.32,37.999826


<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Create Modeling Dataset

</h2>

Create a separate copy of the processed dataset before performing feature selection and model preparation.

In [75]:
d1=df1.copy()

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Remove Redundant Features

</h2>
Drop columns that are no longer required:

- weight
- height
- height_m
- city

These features are either represented through engineered features or replaced by city tier information.

In [76]:
data=d1.drop(['weight','height_m', 'height', 'city',], axis=1, inplace=True)


<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Final Dataset Inspection

</h2>

Review the final dataset that will be used for machine learning.

In [77]:
d1

,age,income_lpa,smoker,occupation,insurance_premium_category,city_tier,bmi,income_per_age,weight_height_ratio,income_bmi,smoker_income,health_risk_index
0,50,29.90,0,Factory Worker,Medium,3,18.401343,0.598000,0.323864,550.200155,0.00,22.360537
1,56,19.22,0,Businessman,Medium,3,31.887755,0.343214,0.535714,612.882653,0.00,29.555102
2,26,36.95,0,Sales Manager,Low,3,27.458654,1.421154,0.488764,1014.597273,0.00,18.783462
3,34,14.75,0,Banker,Low,3,27.815882,0.433824,0.450617,410.284255,0.00,21.326353
4,43,19.32,1,Marketing Manager,High,3,12.749566,0.449302,0.244792,246.321615,19.32,37.999826
...,...,...,...,...,...,...,...,...,...,...,...,...
3995,25,37.65,1,Insurance Agent,Medium,3,23.624447,1.506000,0.382716,889.460448,37.65,36.949779
3996,29,10.00,0,Data Analyst,Low,3,28.360352,0.344828,0.479290,283.603515,0.00,20.044141
3997,37,28.63,1,Chef,High,2,17.258941,0.773784,0.324468,494.123472,28.63,38.003576
3998,26,12.01,0,Pharmacist,Low,2,17.270698,0.461923,0.335052,207.421086,0.00,14.708279


<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Define Features and Target
</h2>

Split the dataset into:

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Features (X)

</h2>

All predictor variables.

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Target (y)

</h2>

Insurance Premium Category:
- Low
- Medium
- High

In [78]:
X = d1.drop("insurance_premium_category", axis=1)
y = d1["insurance_premium_category"]

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Separate Categorical and Numerical Features

</h2>

Identify categorical and numerical columns to apply appropriate preprocessing techniques.

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Categorical Features

</h2>

- Occupation
- City Tier
- Smoker Status

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Numerical Features

</h2>

- Age
- Income
- BMI
- Income per Age
- Weight-to-Height Ratio
- Income × BMI
- Smoker × Income
- Health Risk Index

In [79]:
categorical_features = [
    "occupation",
    "city_tier",
    "smoker"
]

numeric_features = [
    "age",
    "income_lpa",
    "bmi",
    "income_per_age",
    "weight_height_ratio",
    "income_bmi",
    "smoker_income",
    "health_risk_index"
]

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Build Preprocessing Pipeline
</h2>
Use a ColumnTransformer to:

- Apply One-Hot Encoding to categorical features.
- Pass numerical features directly to the model.

This ensures all data is transformed into a machine-learning-friendly format.

In [80]:
preprocessor = ColumnTransformer(
transformers=[
("cat", OneHotEncoder(), categorical_features),
("num", "passthrough", numeric_features)
])

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Train-Test Split
</h2>
Split the dataset into training and testing subsets.

### Purpose
- Training Set → Model Learning
- Testing Set → Model Evaluation

Test Size: 20%

In [81]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Build and Train Random Forest Model
</h2>
Create a machine learning pipeline consisting of:

1. Data Preprocessing
2. Random Forest Classifier

The model is then trained using the training dataset and used to generate predictions.

In [82]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor), 
    ("classifier", RandomForestClassifier(n_estimators=100,random_state=42))
])

pipeline.fit(x_train, y_train)
y_pred = pipeline.predict(x_test)

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;"> Cross-Validation and Performance Analysis
</h2>
Evaluate model stability using 5-Fold Cross Validation.

Metrics Computed:
- Cross Validation Accuracy
- Mean Accuracy
- Training Accuracy
- Testing Accuracy

This helps assess model generalization and detect overfitting.

In [83]:
scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print(scores)
print("Mean Accuracy:", scores.mean())

print("Train Accuracy:", pipeline.score(x_train, y_train))
print("Test Accuracy :", pipeline.score(x_test, y_test))

[0.8925  0.89625 0.8725  0.9075  0.9175 ]
Mean Accuracy: 0.89725
Train Accuracy: 0.9996875
Test Accuracy : 0.89625


<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Accuracy Evaluation
</h2>
Calculate overall classification accuracy on the test dataset.

Accuracy measures the percentage of correctly classified insurance premium categories.

In [84]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

Model Accuracy: 0.90


<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Classification Report
</h2>
Generate detailed classification metrics:

- Precision
- Recall
- F1-Score
- Support

for each insurance premium category.

This provides a deeper understanding of model performance across classes.

In [85]:
classification_rep = classification_report(y_test, y_pred)
print("Classification Report:")
print(classification_rep)

Classification Report:
              precision    recall  f1-score   support

        High       0.91      0.90      0.91       203
         Low       0.91      0.93      0.92       270
      Medium       0.88      0.87      0.87       327

    accuracy                           0.90       800
   macro avg       0.90      0.90      0.90       800
weighted avg       0.90      0.90      0.90       800



<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Sample Prediction
</h2>

Test the trained model using a hypothetical customer profile.

The model predicts the most likely insurance premium category based on the provided attributes.

In [86]:
#sample input for testing
sample_input = {
    "age": 30,
    "income_lpa": 12,
    "occupation": "Engineer",
    "smoker": 1,
    "city_tier": 1,
    "bmi": 24.5,
    "income_per_age": 0.4,
    "weight_height_ratio": 0.5,
    "income_bmi": 294,
    "smoker_income": 12,
    "health_risk_index": 15.8
}

sample_df = pd.DataFrame([sample_input])
sample_pred = pipeline.predict(sample_df)
print(f"Predicted Insurance Premium Category: {sample_pred[0]}")

Predicted Insurance Premium Category: Medium


<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">Save Trained Model</h2>

Serialize and store the trained machine learning pipeline using Pickle.
This allows the model to be reused without retraining.

In [87]:
with open("insurance_premium_model.pkl", "wb") as f:
    dump(pipeline, f)

<h2 style="background: linear-gradient(to right, #4F46E5, #06B6D4); -webkit-background-clip: text; color: transparent;">
Load and Verify Saved Model
</h2>

Load the saved model from disk and perform a prediction to ensure successful serialization and deployment readiness.

In [88]:
with open("insurance_premium_model.pkl", "rb") as f:
    loaded_model = load(f)
    
loaded_pred = loaded_model.predict(sample_df)
print(f"Predicted Insurance Premium Category from Loaded Model: {loaded_pred[0]}")

Predicted Insurance Premium Category from Loaded Model: Medium
